#initilize all the global variables and load all the required pa

In [139]:
import requests
import json
import urllib.request
import time
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
from urllib.request import urlopen
import os
import re
from datetime import date

path="C:/Users/Supra/Documents/Supravat/Documents/ISB/Data Collection"
os.chdir(path)


Years = []
Recipients = []
Years = []
Url = []
States = []
Fields = []
recipient_dict= {}
final_urls= []
attributes = {'Born', 'Died', 'Nationality','Alma mater','Known for' ,'Awards'}
Born = []
Died = []
Nationality = []
Spouse = []
Alma= []
president_dict ={} 
Awarded_By = []
Institutions = []
Awards = []
Occupation = []
ElectionYear= []
dict_election = {}
padma_award_declaration_date = '26 Jan '

#Scrape the wiki page of the president and put the information in a dictionary to find out which
#president has given the award.

In [140]:
def populate_president():
    ttook_office = []
    left_office = []
    
    
    response = requests.get("https://en.wikipedia.org/wiki/List_of_presidents_of_India").text

    soup = BeautifulSoup(response, 'html')
    table = soup.find("table", {"id":"pm_list"})
    rows = table.find_all('tr')
    for row in rows:
 
 

      link = row.find('a')
 
      if link is not None :
    
         prez_name= link.get('title')
    
         tds = row.find_all('td')
  
      if tds is not None :
       if len(tds)> 0 :
        
         took_office = tds[2].text.strip()
         left_office = tds[3].text.strip()
         pos = left_office.find("(")
         if pos > 0 :
           left_office= left_office[:pos]
         if left_office == 'Incumbent' :
            
            today = date.today()
            left_office = today.strftime(" %d %B %Y")
        
         president_dict.update({took_office:[prez_name,left_office]})
     
    print(president_dict)
  


#Scrape the election year page wiki and put it into a dictionary to derive if the year on which Padma 
#award is given is an election year for populist measure 

In [141]:
def populate_election_year():
    i = 0
    response = requests.get("https://en.wikipedia.org/wiki/List_of_Indian_general_elections").text
    soup = BeautifulSoup(response, 'html')
    table = soup.find("table", {"class":"wikitable plainrowheaders"})
    rows = table.find_all('tr')
    #for row in rows:
  
    for row in rows :    

        link = row.find('a')
    
        if link is not None :
    
         election_year= link.get('title')
         dict_election[i] = election_year[0:4]
         i = i +1
    print(dict_election)

In [142]:
# read the each wikipage of the different padmaawards and extract padma awardee information and other details

In [143]:
def  process_records(soup) :
    tables = soup.find_all('table')
    my_table =[]
    final_urls = []
    prez_name = ' '
    recipeint_dict_test = {}
    election_year = ''
    for table in tables :

        my_table = soup.find("table",{"class":"wikitable plainrowheaders sortable"}) 
   
    rows = my_table.find_all('tr') 
           
    for row in rows :
                ths = row.find('th')
                
                if ths :
                 if len(ths)> 0 and ths.get('scope') == 'row' :
                   
                   Recipients.append(ths.text.strip())
                   print('resi' , ths.text.strip())
                
                tds =row.find_all('td')
                if len(tds) == 3 :
                     year = tds[0]
                     Years.append(year.text.strip())
                     
                     prez_name = check_prez_name(str(year.text.strip()))
                     Awarded_By.append(prez_name)
                     election_year = check_election_year(str(year.text.strip()))
                     ElectionYear.append(election_year)
                     field = tds[1]
                     Fields.append(field.text.strip())
                     state = tds[2]
                     States.append(state.text.strip())
                elif len(tds) == 4 :
                      year = tds[0]
                      Years.append(year.text.strip())
                      prez_name = check_prez_name(str(year.text.strip()))
                      Awarded_By.append(prez_name)
                      election_year = check_election_year(str(year.text.strip()))
                      print(election_year)
                      ElectionYear.append(election_year)
                      Recipients.append(tds[1].text.strip())
                      field = tds[2]
                      Fields.append(field.text.strip())
                      state = tds[3]
                      States.append(state.text.strip())
                 
                links= row.find('a')  
                if links :
                    wiki_url = links.get('href')
                    if 'https' not in wiki_url :
                      final_urls= "https://en.wikipedia.org" + wiki_url
                    Url.append(final_urls)
                    process_more_records(final_urls)
                elif  row.find('td')   :
                     #print(row)
                     final_urls = 'not available'
                     Url.append(final_urls)
                     Born_field = 'URL not available'
                     Born.append(Born_field)
                     Died_field ='URL not available'
                     Died.append(Died_field)
                     Nationality.append('URL not available')
                     Spouse.append('URL not available')
                     Alma.append('URL not available')
                     Institutions.append('URL not available')
                     
                     Awards.append('URL not available')
                     Occupation.append('URL not available')
                     


#Do the webscraping of the individual wiki page of the awardee and try to extract fields which 
#are in structured format

In [144]:
def  process_more_records(final_urls):
  info = {}
  tbl = []
  list_of_table_rows = []
  try:
      r = requests.get(final_urls)
      soup = BeautifulSoup(r.text, "lxml")
      tbl = soup.find("table" , {"class": "infobox biography vcard"}) 
      if tbl is None :
        tbl = soup.find("table" , {"class": "infobox vcard"}) 
      if tbl is None :
        tbl = soup.find("table" , {"class": "infobox vcard plainlist"}) 
      if tbl :
        list_of_table_rows = tbl.findAll('tr')
  
        for tr in list_of_table_rows:

            th = tr.find("th")
            td = tr.find("td")
            if th is not None:
                innerText = ''
                if td :
                  for elem in td.recursiveChildGenerator():
                    if isinstance(elem, str):
                        innerText += elem.strip()
                    elif elem.name == 'br':
                        innerText += '\n'
                info[th.text] = innerText  
  except Exception as E:
           info= ''
    
  #the page of Awardee does not have any strucuted information for personal data
  if tbl is None:
       info =' '  
  populate_df_extra_fields(info)   

     

#Populate the fields which can be picked up from the individual wiki pages of the awardee

In [145]:
def populate_df_extra_fields(info) :
       Born_field = []
       Died_field = []
       Spouse_field = []
       Nationality_field= []
       Alma_field = []
       Institutions_field = []
       Awards_field = []
       if 'Born' in info :
            Born_field = info['Born']
       else:
       
            Born_field = 'n/a'
       Born.append(Born_field)
       if 'Died' in info :
            Died_field = info['Died']
       else:
           Died_field= 'n/a'
       Died.append(Died_field)
       if 'Nationality' in info :
             Nationality_field = info['Nationality']
       else :
             Nationality_field = 'n/a'
       Nationality.append(Nationality_field)
       if 'Spouse'  in info :
             Spouse_field = info['Spouse']
       elif 'Spouse(s)' in info :
             Spouse_field = info['Spouse(s)']
       else :
             Spouse_field = 'n/a'
       Spouse.append(Spouse_field)
    
       if 'Alma mater' in info :
             Alma_field = info['Alma mater']
       else :
             Alma_field = 'n/a'
       Alma.append(Alma_field)
       if 'Institutions' in info:
            Institutions_field = info['Institutions']
            
       else :
            Institutions_field = "n/a"
            
       Institutions.append(Institutions_field)
       if ('Awards' in info) :
             Awards_field = info['Awards']
       elif  ('Notable awards' in info):
             Awards_field = info['Notable awards']
       else :
             Awards_field = 'n/a'
       Awards.append(Awards_field)
       if 'Occupation' in info:
            Occupation.append(info['Occupation'])
       else:
            Occupation.append('n/a')
          

#Populate which president has given awards on that particular year

In [146]:
def check_prez_name(year) :
    Year = ' '
    date = padma_award_declaration_date
    prez_name = ''
 
    date_award = date + str(year)

    for key,value in president_dict.items():
     if (pd.to_datetime(date_award) > pd.to_datetime(key))  :
      if (pd.to_datetime(date_award) < pd.to_datetime(value[1])) :
         prez_name = value[0]
         
    return(prez_name)
    
    


#Check if the year on which award is given is an election year or not

In [147]:
def check_election_year(year) :
    flag_election= 'No'
    for key,value in  dict_election.items() :
        if year == value : 
            flag_election= 'Yes'
            continue
    return flag_election  

In [148]:
def write_to_csv(AwardType):

   
    df_padmasree = pd.DataFrame(Recipients, index= Recipients ,
    columns = ['Recipients'])
    

    df_padmasree['Year'] = Years
    if AwardType == 'PadmaShri' : 
         df_padmasree['Award Type'] = 'PadmaShri'
    
    if AwardType == 'PadmaVibhushan' :
         df_padmasree['Award Type'] = 'PadmaVibhushan'
    
    if AwardType == 'PadmaBhushan':
         df_padmasree['Award Type'] = 'PadmaBushan'
                    

    df_padmasree['State'] = States
    df_padmasree['Field'] = Fields
          
    df_padmasree['WikiLink'] = Url
    df_padmasree['Born'] = Born
    df_padmasree['Died'] = Died
    df_padmasree['Nationality'] = Nationality
    df_padmasree['Spouse'] = Spouse
    df_padmasree['Alma mater'] =Alma
    df_padmasree['Institutions'] = Institutions
    df_padmasree['Other Awards including Padma'] = Awards
    df_padmasree['Awarded By'] = Awarded_By
    df_padmasree['Occupation'] = Occupation
    df_padmasree['Election Year'] = ElectionYear
    if AwardType == 'PadmaShri' : 
         df_padmasree.to_csv('PadmaShri.csv', index=False)

    if AwardType == 'PadmaVibhushan' :
        df_padmasree.to_csv('PadmaVibhushan.csv', index=False)
    
    if AwardType == 'PadmaBhushan':
        df_padmasree.to_csv('PadmaBhushan.csv', index=False)
    




In [149]:
#Crawling for Padma Awardee List from Wiki

In [150]:
def padma_crawl(AwardType):
    urlinfo = []
    existing_links = {}
    if AwardType == 'PadmaShri' :
        pos = AwardType.find('Shri')
    elif AwardType == 'PadmaBhushan' :
        pos = AwardType.find('Bhushan')
    elif AwardType == 'PadmaVibhushan' :
        pos = AwardType.find('Vibhushan')
    UrlText = AwardType[ :pos] +'_' +AwardType[pos:]
    urllink = urllink1 = 'https://en.wikipedia.org/wiki/' + UrlText
    urllink2 = '^/wiki/List_of_' + UrlText
   
    response = requests.get(urllink1).text
    
    soup = BeautifulSoup(response, 'html')

    for link in soup.find_all('a',attrs={'href':re.compile(urllink2)}):
        S=link.get('href')
   
        if S not in existing_links:
             urlinfo.append(("https://en.wikipedia.org""%s\n" % S).strip())
             existing_links[S] = True
    print(urlinfo)
    return(urlinfo)

In [151]:
def process_main(AwardType) :
 urllist = []
 urllist = padma_crawl(AwardType)
 print(urllist)
 url = ' '
 
 
# Populate information of Presidents who took office and demitted office and load in a dictionary
 populate_president()
 # Find out the election year on which elections were held   
 populate_election_year()
 # read the seed urls of the Awardee and write the exracted information in a CSV file
 for url in urllist :
    print(url)
    html = urlopen(url) 
    soup = BeautifulSoup(html, 'html.parser')
    
    process_records(soup)
    write_to_csv(AwardType)


In [152]:
#Read and process the Wiki Page of the Padma Awardee based on the input recived from the user 

In [153]:
AwardType = input("Enter the Award Type['PadmaVibhushan', 'PadmaBhushan', 'PadmaShri']: ")
if AwardType not in ['PadmaVibhushan', 'PadmaBhushan', 'PadmaShri'] :
     print('please enter correct Award Type')
else :
    print('Data Collection for ',AwardType ,' started')
    process_main(AwardType) 


Enter the Award Type['PadmaVibhushan', 'PadmaBhushan', 'PadmaShri']: PadmaVibhushan
Data Collection for  PadmaVibhushan  started
['https://en.wikipedia.org/wiki/List_of_Padma_Vibhushan_award_recipients']
['https://en.wikipedia.org/wiki/List_of_Padma_Vibhushan_award_recipients']
{'26 January 1950': ['Rajendra Prasad', '13 May 1962'], '13 May 1962': ['Sarvepalli Radhakrishnan', '13 May 1967'], '13 May 1967': ['Zakir Husain (politician)', '3 May 1969'], '3 May  1969': ['V. V. Giri', '20 July 1969'], '20 July 1969': ['Mohammad Hidayatullah', '24 August 1969'], '24 August 1969': ['V. V. Giri', '24 August 1974'], '24 August 1974': ['Fakhruddin Ali Ahmed', '11 February 1977'], '11 February 1977': ['B. D. Jatti', '25 July 1977'], '25 July 1977': ['Neelam Sanjiva Reddy', '25 July 1982'], '25 July 1982': ['Zail Singh', '25 July 1983'], '25 July 1983': ['Zail Singh', '25 July 1984'], '25 July 1984': ['Zail Singh', '25 July 1987'], '25 July 1987': ['R. Venkataraman', '25 July 1992'], '25 July 1992

resi Ramoji Rao
resi Sri Sri Ravi Shankar[k]
resi V. Shanta
resi Murli Manohar Joshi
resi Sunder Lal Patwa[xi]#
resi Sharad Pawar
resi Udupi Ramachandra Rao
resi P. A. Sangma[xii]#
resi Jaggi Vasudev
resi K. J. Yesudas
resi Ilaiyaraaja
resi Ghulam Mustafa Khan
resi P. Parameswaran
resi Teejan Bai
resi Ismaïl Omar Guelleh
resi Anil Manibhai Naik
resi Balwant Moreshwar Purandare
resi George Fernandes[xiii]#
resi Arun Jaitley[xiv]#
resi Anerood Jugnauth
resi Mary Kom
resi Chhannulal Mishra
resi Sushma Swaraj[xv]#
resi Vishwesha Teertha[xvi]#
resi Shinzo Abe
resi S. P. Balasubramaniam[xvii]#
resi Belle Monappa Hegde
resi Narinder Singh Kapany[xviii]#
resi Wahiduddin Khan
resi B. B. Lal
resi Sudarshan Sahoo
